# Revisão: KNN - K-Vizinhos Mais Próximos
Classificação por proximidade: o algoritmo consulta os exemplos mais parecidos para decidir.

Esta revisão cobre:
- Distância euclidiana (na mão)
- KNN manual sem biblioteca
- KNN com scikit-learn
- Testando valores de K diferentes

In [ ]:
# math: raiz quadrada para calcular distancia
import math

# pandas: tabelas de dados
import pandas as pd

# KNeighborsClassifier: KNN do scikit-learn
from sklearn.neighbors import KNeighborsClassifier

## Distância Euclidiana
O KNN decide 'quem está mais perto' usando a distância euclidiana:
a mesma fórmula do Teorema de Pitágoras.

$$d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$$

Exemplo: dois clientes com renda e histórico de pagamento (escala de 0 a 10).

In [ ]:
# Cliente A e cliente B com 2 features: renda e historico de pagamento
p1 = (2.0, 3.0)  # cliente A: renda baixa, historico ruim
p2 = (8.0, 7.0)  # cliente B: renda alta, historico bom

# Diferenca em cada dimensao (cateto do triangulo retangulo)
dx = p1[0] - p2[0]
dy = p1[1] - p2[1]

# Distancia = hipotenusa (Pitagoras)
distancia = math.sqrt(dx**2 + dy**2)

print(f"Cliente A: renda={p1[0]}, historico={p1[1]}")
print(f"Cliente B: renda={p2[0]}, historico={p2[1]}")
print(f"Distancia entre eles: {distancia:.2f}")

In [ ]:
# Comparacao simples: distancia Euclidiana vs Manhattan
p1 = (2.0, 3.0)
p2 = (8.0, 7.0)

dx = p1[0] - p2[0]
dy = p1[1] - p2[1]

dist_euclidiana = math.sqrt(dx**2 + dy**2)
dist_manhattan = abs(dx) + abs(dy)

print(f"Euclidiana: {dist_euclidiana:.2f}")
print(f"Manhattan: {dist_manhattan:.2f}")
print()
print("Euclidiana: distancia em linha reta.")
print("Manhattan: soma dos passos em grade (horizontal + vertical).")

## Distância para Vários Pontos
Na prática, calculamos a distância de um novo ponto para TODOS os pontos do dataset.
Depois ordenamos e pegamos os K mais próximos.

In [ ]:
# Clientes conhecidos com resultado real
clientes = [
    {"renda": 8.0, "historico": 9.0, "aprovado": 1},
    {"renda": 3.0, "historico": 2.0, "aprovado": 0},
    {"renda": 6.0, "historico": 7.0, "aprovado": 1},
    {"renda": 2.0, "historico": 5.0, "aprovado": 0},
    {"renda": 9.0, "historico": 8.0, "aprovado": 1},
]

# Novo cliente: queremos prever se sera aprovado
novo = (5.0, 6.0)

# Calcula a distancia do novo para cada cliente conhecido
for c in clientes:
    dx = novo[0] - c["renda"]
    dy = novo[1] - c["historico"]
    dist = math.sqrt(dx**2 + dy**2)
    print(f"  dist={dist:.2f} | renda={c['renda']}, hist={c['historico']}, aprovado={c['aprovado']}")

## KNN Manual — Votação
Com as distâncias calculadas, ordenamos do mais próximo ao mais distante
e deixamos os K vizinhos mais próximos votarem.

A classe com mais votos vence.

In [ ]:
k = 3  # numero de vizinhos que vao votar

# Calcula distancia e guarda junto com o resultado
clientes_com_dist = []
for c in clientes:
    dx = novo[0] - c["renda"]
    dy = novo[1] - c["historico"]
    dist = math.sqrt(dx**2 + dy**2)
    clientes_com_dist.append({"dist": dist, **c})

# Ordena do mais proximo para o mais distante
clientes_com_dist.sort(key=lambda x: x["dist"])

# Pega os K primeiros (os mais proximos)
vizinhos = clientes_com_dist[:k]
votos = [v["aprovado"] for v in vizinhos]

print(f"Novo cliente: renda={novo[0]}, historico={novo[1]}")
print(f"\nOs {k} vizinhos mais proximos:")
for v in vizinhos:
    label = "Aprovado" if v["aprovado"] == 1 else "Reprovado"
    print(f"  dist={v['dist']:.2f} -> {label}")

# Votacao: maioria decide
resultado = 1 if sum(votos) > k // 2 else 0
print(f"\nVotos: {votos}")
print(f"Previsao: {'Aprovado' if resultado == 1 else 'Reprovado'}")

## KNN com Scikit-learn
O scikit-learn faz tudo isso automaticamente com `KNeighborsClassifier`.

O fluxo é sempre o mesmo:
1. Criar o modelo
2. `fit()` com os dados de treino
3. `predict()` com novos dados

Além disso, o parâmetro `metric` define a distância usada no KNN.
Para testar, troque facilmente entre `"euclidean"` e `"manhattan"`.

In [ ]:
# Dataset de treino: mesmos clientes de antes + mais alguns
df = pd.DataFrame([
    {"renda": 8.0, "historico": 9.0, "aprovado": 1},
    {"renda": 3.0, "historico": 2.0, "aprovado": 0},
    {"renda": 6.0, "historico": 7.0, "aprovado": 1},
    {"renda": 2.0, "historico": 5.0, "aprovado": 0},
    {"renda": 9.0, "historico": 8.0, "aprovado": 1},
    {"renda": 1.0, "historico": 1.0, "aprovado": 0},
    {"renda": 7.0, "historico": 6.0, "aprovado": 1},
    {"renda": 4.0, "historico": 3.0, "aprovado": 0},
])

# Separa features (X) e rotulo (y)
X = df[["renda", "historico"]]
y = df["aprovado"]

# Troque aqui para testar outra distancia: "euclidean" ou "manhattan"
metrica = "euclidean"

# Cria e treina o modelo com K=3
modelo = KNeighborsClassifier(n_neighbors=3, metric=metrica)
modelo.fit(X, y)

# Novo cliente para prever
novo_cliente = pd.DataFrame([{"renda": 5.0, "historico": 6.0}])
previsao = modelo.predict(novo_cliente)

print(f"Metrica usada: {metrica}")
print(f"Novo cliente: renda=5.0, historico=6.0")
print(f"Previsao (K=3): {'Aprovado' if previsao[0] == 1 else 'Reprovado'}")

## Testando Valores de K
O valor de K muda o comportamento do modelo:
- K muito baixo (K=1): sensível a ruído
- K muito alto: ignora detalhes locais

Convença-se testando o mesmo cliente com K diferentes.

In [ ]:
# Testa o mesmo cliente com K = 1, 3 e 5
metrica = "euclidean"  # altere para "manhattan" e compare
for k in [1, 3, 5]:
    m = KNeighborsClassifier(n_neighbors=k, metric=metrica)
    m.fit(X, y)
    prev = m.predict(novo_cliente)[0]
    label = "Aprovado" if prev == 1 else "Reprovado"
    print(f"metric={metrica} | K={k} = {label}")

---
## Resumo
| Conceito | O que faz |
|---|---|
| Distância euclidiana | Mede o quão parecidos são dois pontos |
| K vizinhos | Quantos exemplos votam na decisão |
| fit() | Memoriza os dados de treino |
| predict() | Calcula distâncias e vota |

No próximo notebook: por que normalizar os dados é obrigatório no KNN,
e como usar KNN para recomendação por similaridade.